Notebook 01: data cleaning

Objective: goal is to clean and prepare the raw video game sales dataset for analysis

Focus: 
- inspecting the raw dataset
- identifying missing or invalid values
- removing columns that do not add analytical value
- cleaning year and sales fields
- creating new features for later analysis
- exporting a clean dataset for SQL, visualization, and machine learning notebooks

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
raw_path = Path("../data/raw/vgsales.csv")

df_raw = pd.read_csv(raw_path)

df_raw.head()


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [3]:
df_raw.shape

(16598, 11)

In [4]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


In [5]:
df_raw.describe(include="all")

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16598,16598,16327.000000,16598,16540,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
unique,NaN,11493,31,NaN,12,578,NaN,NaN,NaN,NaN,NaN
top,NaN,Need for Speed: Most Wanted,DS,NaN,Action,Electronic Arts,NaN,NaN,NaN,NaN,NaN
freq,NaN,12,2163,NaN,3316,1351,NaN,NaN,NaN,NaN,NaN
mean,8300.605254,NaN,NaN,2006.406443,NaN,NaN,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,NaN,NaN,5.828981,NaN,NaN,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,NaN,NaN,1980.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,NaN,NaN,2003.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,NaN,NaN,2007.000000,NaN,NaN,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,NaN,NaN,2010.000000,NaN,NaN,0.240000,0.110000,0.040000,0.040000,0.470000


In [6]:
df_raw.columns

Index(['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
      dtype='str')

In [7]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['rank', 'name', 'platform', 'year', 'genre', 'publisher', 'na_sales',
       'eu_sales', 'jp_sales', 'other_sales', 'global_sales'],
      dtype='str')

In [8]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

missing_summary

,missing_count,missing_percent
year,271,1.632727
publisher,58,0.349440
rank,0,0.000000
platform,0,0.000000
name,0,0.000000
genre,0,0.000000
na_sales,0,0.000000
eu_sales,0,0.000000
jp_sales,0,0.000000
other_sales,0,0.000000


In [9]:
df[df["year"].isna()]

,rank,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales
179,180,Madden NFL 2004,PS2,NaN,Sports,Electronic Arts,4.26,0.26,0.01,0.71,5.23
377,378,FIFA Soccer 2004,PS2,NaN,Sports,Electronic Arts,0.59,2.36,0.04,0.51,3.49
431,432,LEGO Batman: The Videogame,Wii,NaN,Action,Warner Bros. Interactive Entertainment,1.86,1.02,0.00,0.29,3.17
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,NaN,1.57,1.02,0.00,0.41,3.00
607,608,Space Invaders,2600,NaN,Shooter,Atari,2.36,0.14,0.00,0.03,2.53
624,625,Rock Band,X360,NaN,Misc,Electronic Arts,1.93,0.34,0.00,0.21,2.48
649,650,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,Konami Digital Entertainment,2.15,0.18,0.00,0.07,2.39
652,653,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,LucasArts,1.54,0.63,0.00,0.22,2.39
711,713,Call of Duty 3,Wii,NaN,Shooter,Activision,1.19,0.84,0.00,0.23,2.26
782,784,Rock Band,Wii,NaN,Misc,MTV Games,1.35,0.56,0.00,0.20,2.11


Since the `year` column is important because several analysis questions depend on time trends, platform lifecycles, and release periods. Rows with missing release years cannot be accurately placed on a timeline, so they will be removed from the cleaned dataset.

In [10]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(0)

Duplicate rows were checked to avoid double-counting games during aggregation. If duplicates were found, they were removed.

In [11]:
df.dtypes

,0
rank,int64
name,str
platform,str
year,float64
genre,str
publisher,str
na_sales,float64
eu_sales,float64
jp_sales,float64
other_sales,float64


In [12]:
df["year"].describe()

,year
count,16327.000000
mean,2006.406443
std,5.828981
min,1980.000000
25%,2003.000000
50%,2007.000000
75%,2010.000000
max,2020.000000


In [13]:
df["year"].value_counts().sort_index()

,count
year,
1980.0,9
1981.0,46
1982.0,36
1983.0,17
1984.0,14
1985.0,14
1986.0,21
1987.0,16
1988.0,15


In [14]:
rows_before_year_cleaning = df.shape[0]

df = df.dropna(subset=["year"])

rows_after_missing_year_drop = df.shape[0]

rows_removed_missing_year = rows_before_year_cleaning - rows_after_missing_year_drop
rows_removed_missing_year

271

Rows with missing `publisher` values are removed because publisher is required for publisher-level analysis. Assigning `"Unknown"` would create an artificial publisher category and could distort publisher comparisons. Only 36 rows are affected, so this removal should have a negligible effect on the overall dataset.

In [15]:
rows_before = len(df)
shape_before_publisher_drop = df.shape
missing_publishers = df["publisher"].isna().sum()

df = df.dropna(subset=["publisher"]).copy()
df = df.reset_index(drop=True)

rows_after = len(df)
shape_after_publisher_drop = df.shape
rows_removed_missing_publishers = rows_before - rows_after


In [16]:
df["year"] = df["year"].astype(int)

In [17]:
rows_before_2016_filter = df.shape[0]

df = df[df["year"] <= 2016]

rows_after_2016_filter = df.shape[0]

rows_removed_after_2016 = rows_before_2016_filter - rows_after_2016_filter
rows_removed_after_2016

4

In [18]:
df["year"].min(), df["year"].max()

(np.int64(1980), np.int64(2016))

Rows with missing release years were removed because they cannot support time-based analysis.  
Rows with missing publishers were removed because publisher is required for publisher-level analysis and assigning `"Unknown"` would create an artificial publisher category.  
Rows after 2016 were also removed because the dataset appears incomplete in later years, which could distort release trends and platform lifecycle analysis.

In [19]:
if "rank" in df.columns:
    df = df.drop(columns=["rank"])

The `rank` column was removed because it is a precomputed ordering based on global sales. Since global sales already exists in the dataset, rank does not add new information and could bias later analysis.

In [20]:
sales_cols = ["na_sales", "eu_sales", "jp_sales", "other_sales", "global_sales"]

for col in sales_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    
df[sales_cols].isna().sum()

,0
na_sales,0
eu_sales,0
jp_sales,0
other_sales,0
global_sales,0


In [21]:
(df[sales_cols] < 0).sum()

,0
na_sales,0
eu_sales,0
jp_sales,0
other_sales,0
global_sales,0


In [22]:
df["regional_sales_total"] = (
    df["na_sales"] + df["eu_sales"] + df["jp_sales"] + df["other_sales"]
)

df["sales_difference"] = df["global_sales"] - df["regional_sales_total"]

df[["global_sales", "regional_sales_total", "sales_difference"]].describe()

,global_sales,regional_sales_total,sales_difference
count,16287.000000,16287.000000,16287.000000
mean,0.541022,0.540749,0.000273
std,1.567520,1.567636,0.005225
min,0.010000,0.000000,-0.020000
25%,0.060000,0.060000,0.000000
50%,0.170000,0.170000,0.000000
75%,0.480000,0.480000,0.000000
max,82.740000,82.740000,0.020000


In [23]:
df = df.drop(columns=["sales_difference"])

Regional sales were compared against global sales as a validation step. Small differences are expected because sales values are rounded to millions.

In [24]:
df["decade"] = (df["year"] // 10) * 10
df["decade"] = df["decade"].astype(str) + "s"
df[["year", "decade"]].head()

,year,decade
0,2006,2000s
1,1985,1980s
2,2008,2000s
3,2009,2000s
4,1996,1990s


In [25]:
def assign_sales_tier(global_sales):
    if global_sales >= 5:
        return "blockbuster"
    elif global_sales >= 1:
        return "high"
    elif global_sales >= 0.5:
        return "medium"
    else:
        return "low"

df["sales_tier"] = df["global_sales"].apply(assign_sales_tier)

df["sales_tier"].value_counts()

,count
sales_tier,
low,12327
medium,1902
high,1852
blockbuster,206


A `sales_tier` feature was created to group games into low, medium, high, and blockbuster sales categories. This will be useful for later classification modeling and for comparing performance across genres, platforms, and publishers.

In [26]:
df["is_high_seller"] = np.where(df["global_sales"] >= 1, 1, 0)

In [27]:
df["is_high_seller"].value_counts(normalize=True) * 100

,proportion
is_high_seller,
0,87.364155
1,12.635845


A binary `is_high_seller` column was created for future machine learning analysis. Games with at least 1 million global sales are labeled as high sellers.

In [28]:
regional_total = df["regional_sales_total"]

df["na_sales_share"] = np.where(
    regional_total > 0,
    df["na_sales"] / regional_total,
    0
)

df["eu_sales_share"] = np.where(
    regional_total > 0,
    df["eu_sales"] / regional_total,
    0
)

df["jp_sales_share"] = np.where(
    regional_total > 0,
    df["jp_sales"] / regional_total,
    0
)

df["other_sales_share"] = np.where(
    regional_total > 0,
    df["other_sales"] / regional_total,
    0
)

In [29]:
share_columns = [
    "na_sales_share",
    "eu_sales_share",
    "jp_sales_share",
    "other_sales_share"
]

share_totals = df[share_columns].sum(axis=1)

print(share_totals.describe())

regional_share_validation_passed = bool(
    np.allclose(
        share_totals[df["regional_sales_total"] > 0],
        1.0
    )
    and (
        share_totals[df["regional_sales_total"] == 0] == 0
    ).all()
)

assert np.allclose(
    share_totals[df["regional_sales_total"] > 0],
    1.0
)

assert (
    share_totals[df["regional_sales_total"] == 0] == 0
).all()

regional_share_validation_passed

count    16287.000000
mean         0.999939
std          0.007836
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
dtype: float64


True

In [30]:
region_sales_cols = [
    "na_sales",
    "eu_sales",
    "jp_sales",
    "other_sales"
]

df["dominant_region"] = df[region_sales_cols].idxmax(axis=1)

region_name_map = {
    "na_sales": "North America",
    "eu_sales": "Europe",
    "jp_sales": "Japan",
    "other_sales": "Other"
}

df["dominant_region"] = df["dominant_region"].map(region_name_map)

no_regional_sales = df[region_sales_cols].sum(axis=1) == 0

df.loc[
    no_regional_sales,
    "dominant_region"
] = "No reported regional sales"

In [31]:
df["dominant_region"].value_counts()

,count
dominant_region,
North America,9897
Japan,3981
Europe,2332
Other,76
No reported regional sales,1


The `dominant_region` feature identifies which region contributed the largest share of reported regional sales for each game. Games with no reported regional sales are labeled `"No reported regional sales"` instead of being assigned to a region by default.

In [32]:
df.shape

(16287, 19)

In [33]:
df.shape

(16287, 19)

In [34]:
df.head()

,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,regional_sales_total,decade,sales_tier,is_high_seller,na_sales_share,eu_sales_share,jp_sales_share,other_sales_share,dominant_region
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,82.74,2000s,blockbuster,1,0.501450,0.350737,0.045564,0.102248,North America
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,40.24,1980s,blockbuster,1,0.722664,0.088966,0.169235,0.019135,North America
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,35.83,2000s,blockbuster,1,0.442367,0.359475,0.105777,0.092381,North America
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,33.00,2000s,blockbuster,1,0.477273,0.333636,0.099394,0.089697,North America
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,31.38,1990s,blockbuster,1,0.359146,0.283301,0.325685,0.031867,North America


In [35]:
df.describe()

,year,na_sales,eu_sales,jp_sales,other_sales,global_sales,regional_sales_total,is_high_seller,na_sales_share,eu_sales_share,jp_sales_share,other_sales_share
count,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000,16287.000000
mean,2006.402775,0.265695,0.147768,0.078849,0.048437,0.541022,0.540749,0.126358,0.459805,0.231793,0.244144,0.064196
std,5.830382,0.822525,0.509360,0.311916,0.190105,1.567520,1.567636,0.332263,0.345459,0.253823,0.402564,0.078392
min,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000,0.060000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000,0.170000,0.000000,0.500000,0.200000,0.000000,0.055556
75%,2010.000000,0.240000,0.110000,0.040000,0.040000,0.480000,0.480000,0.000000,0.750000,0.375000,0.363636,0.100000
max,2016.000000,41.490000,29.020000,10.220000,10.570000,82.740000,82.740000,1.000000,1.000000,1.000000,1.000000,1.000000


In [36]:
cleaning_summary = pd.DataFrame({
    "step": [
        "Original rows",
        "Rows after removing missing years",
        "Rows after removing missing publishers",
        "Rows after filtering to 2016 and earlier",
        "Final rows",
        "Final columns"
    ],
    "value": [
        df_raw.shape[0],
        rows_after_missing_year_drop,
        rows_after,
        rows_after_2016_filter,
        df.shape[0],
        df.shape[1]
    ]
})

cleaning_summary

,step,value
0,Original rows,16598
1,Rows after removing missing years,16327
2,Rows after removing missing publishers,16291
3,Rows after filtering to 2016 and earlier,16287
4,Final rows,16287
5,Final columns,19


In [37]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

In [38]:
final_dataset_shape = df.shape
final_missing_values = df.isna().sum()
total_missing_values = final_missing_values.sum()
duplicate_row_count = df.duplicated().sum()
year_range = (df["year"].min(), df["year"].max())
negative_sales_remaining = (df[sales_cols] < 0).any().any()
no_reported_regional_sales_count = (
    df["dominant_region"] == "No reported regional sales"
).sum()

print(f"Final dataset shape: {final_dataset_shape}")
print(f"Total missing values: {total_missing_values}")
print(f"Duplicate row count: {duplicate_row_count}")
print(f"Year range: {year_range[0]} to {year_range[1]}")
print(f"Negative sales values remain: {negative_sales_remaining}")
print(f"Regional share validation passed: {regional_share_validation_passed}")
print(
    "Rows labeled No reported regional sales: "
    f"{no_reported_regional_sales_count}"
)

if total_missing_values == 0:
    print("Validation passed: no missing values remain.")
else:
    print("Validation failed: missing values remain.")
    display(final_missing_values[final_missing_values > 0])

assert total_missing_values == 0, "Missing values remain in the cleaned dataset."
assert duplicate_row_count == 0, "Duplicate rows remain in the cleaned dataset."
assert not negative_sales_remaining, "Negative sales values remain in the cleaned dataset."
assert regional_share_validation_passed, "Regional sales share validation failed."

Final dataset shape: (16287, 19)
Total missing values: 0
Duplicate row count: 0
Year range: 1980 to 2016
Negative sales values remain: False
Regional share validation passed: True
Rows labeled No reported regional sales: 1
Validation passed: no missing values remain.


In [39]:
cleaned_file_path = processed_path / "video_game_sales_cleaned.csv"

df.to_csv(cleaned_file_path, index=False)

cleaned_file_path

WindowsPath('../data/processed/video_game_sales_cleaned.csv')

In [40]:
df_cleaned_test = pd.read_csv(cleaned_file_path)
df_cleaned_test.head()

,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,regional_sales_total,decade,sales_tier,is_high_seller,na_sales_share,eu_sales_share,jp_sales_share,other_sales_share,dominant_region
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,82.74,2000s,blockbuster,1,0.501450,0.350737,0.045564,0.102248,North America
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,40.24,1980s,blockbuster,1,0.722664,0.088966,0.169235,0.019135,North America
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,35.83,2000s,blockbuster,1,0.442367,0.359475,0.105777,0.092381,North America
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,33.00,2000s,blockbuster,1,0.477273,0.333636,0.099394,0.089697,North America
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,31.38,1990s,blockbuster,1,0.359146,0.283301,0.325685,0.031867,North America


Cleaning Decisions Summary

The following cleaning and preparation steps were completed:

1. Standardized column names for easier use in Python and SQL.
2. Removed rows with missing release years because they cannot be used in time-based analysis.
3. Removed 36 rows with missing publishers because publisher is required for publisher-level analysis and `"Unknown"` would create an artificial publisher category.
4. Filtered the dataset to include games released up to 2016 because later years appear incomplete.
5. Removed the `rank` column because it is derived from global sales and does not add independent analytical value.
6. Checked duplicate rows to prevent double-counting.
7. Validated sales columns to ensure they were numeric and non-negative.
8. Compared regional sales totals against global sales to check consistency.
9. Calculated regional sales share columns using `regional_sales_total` as the denominator.
10. Labeled games with no regional sales as `"No reported regional sales"` in `dominant_region`.
11. Confirmed that no missing values remain in the cleaned dataset.
12. Created new features:
   - `decade`
   - `sales_tier`
   - `is_high_seller`
   - `regional_sales_total`
   - four regional sales share columns
   - `dominant_region`

The cleaned dataset is now ready for SQL analysis, exploratory data analysis, visualization, and machine learning.